## **Awalé Boissons**

### **1. Problème principal du client**

Déterminer où Awalé Boissons doit investir ses prochains 15 millions FCFA de budget marketing sur les deux prochains trimestres.

De plus, la recommandation doit s'appuyer sur les données disponibles, expliciter les limites de preuve et proposer un dispositif de suivi à 90 jours.

### **2. Décideur**

Aïcha, Marketing Manager d'Awalé Boissons.

Aïcha doit pouvoir utiliser le reporting pour répartir le budget publicitaire entre les différents canaux marketing.

### **3. Utilisateurs opérationnels**

#### **3.1 Community manager**

Responsable de la collecte et du dépôt mensuel des différentes sources nécessaires au reporting.

#### **3.2 Directeur des comptes**

Responsable de la supervision du cycle de reporting, de l'interprétation des résultats et de la validation de la recommandation.

#### **3.3 Marketing Manager**

Consommatrice (Aïcha) finale du reporting et responsable de la décision budgétaire.

### **4. Questions métier**

- Où l'argent marketing est-il dépensé ?
- Comment les ventes évoluent-elles dans le temps, par produit, commune, point de vente et canal de vente ?
- Que disent les clients dans les commentaires et les commandes WhatsApp ?
- Quels enseignements peuvent raisonnablement orienter l'allocation des prochains 15 millions FCFA ?
- Comment reproduire cette analyse chaque mois avec un minimum d'intervention humaine ?

### **5. KPI (indicateure clé de performance) principaux**

Nous allons nous baser sur plusieurs KPI : 
- **Spend marketing** : Montant de dépenses marketing observé par canal et par période.
- **Part du budget par canal** : Part des dépenses marketing totales consacrée à chaque canal.
- **Chiffre d'Affaire (CA)** : Chiffre d'affaires après prise en compte des retours.
- **Evolution du CA** : Variation du CA net entre périodes comparables (avec les valeurs manquantes)
- **Unités vendues** : Nombre d'unités vendues par produit, point de vente, commune et canal de vente.
- **Mix produit** : Répartition des ventes par produit et format.
- **Sentiment client** : Répartition des commentaires analysés en positif, neutre et négatif.
- **Taux de réachat livraison** : Part des clients WhatsApp ayant reçu au moins une commande livrée qui en ont reçu au moins deux.

### **6. KPI supportés par les données actuelles**

Statut des 8 KPI listés en section 5, au niveau des marts actuels (`dbt/models/marts/`) :

| KPI | Statut | Table | Limite |
|---|---|---|---|
| Spend marketing | Supporté | `mart_budget_allocation`, `mart_channel_evidence` | Deux sources (export campagne / media plan) ne se réconcilient pas totalement — l'écart est affiché, pas masqué. |
| Part du budget par canal | Supporté | `mart_budget_allocation` | Repose sur l'export campagne (règle de source ci-dessous) : les canaux saisis à la main y sont sous-représentés. |
| Chiffre d'affaires (CA net) | Supporté | `mart_sales_monthly` | 14 jours de ventes manquants en avril, exclus des taux plutôt qu'assimilés à zéro. |
| Évolution du CA | Supporté | `mart_sales_monthly`, `mart_monthly_performance` | Idem — comparaisons à interpréter avec la couverture de données. |
| Unités vendues | Supporté | `mart_sales_monthly` | — |
| Mix produit | Supporté | `mart_product_mix_monthly` (ventes POS), `mart_whatsapp_monthly` (WhatsApp), `mart_social_monthly` (mentions) | Les 5 SKU POS (BIS-1L, BIS-33, BOU-1L, GIN-1L, GIN-33) sont traduits en produit, format et litres par `int_product_dimension` ; un SKU inconnu fait échouer un test au lieu d'être ignoré. Les ventes POS n'incluent pas les commandes WhatsApp, dont le mix est lu dans le texte des commandes (format parfois absent). |
| Sentiment client | Supporté | `mart_social_monthly` | Classification IA évaluée sur un échantillon de 50 commentaires (voir `docs/ai_documentation.ipynb`), pas une vérité terrain à 100 %. |
| Taux de réachat livraison | Supporté | `mart_whatsapp_customers` | `customer_key` est un téléphone normalisé, pas une identité vérifiée : deux personnes partageant un numéro comptent pour un seul client. Sur la période observée : 389 clients WhatsApp identifiés, dont 313 avec au moins une commande livrée ; 180 ont au moins deux commandes livrées, soit un taux de réachat de **57,5 %**. Seules les commandes livrées comptent : l'ancienne définition (tous statuts) donnait 74,3 % et surestimait le réachat, car elle comptait des commandes annulées ou en cours. |

La définition SQL de chacun de ces KPI (modèle et expression) est dans `docs/semantic_layer.yml` ; un test automatique vérifie que ses définitions sont identiques, mot pour mot, à celles de la section 5 ci-dessus.

Aucun KPI de cette liste n'est aujourd'hui totalement hors de portée des données — le seul point structurellement hors de portée est l'attribution causale d'une vente à un canal (section suivante), qui n'est pas un KPI mais une limite de preuve.

### **6 bis. Règle de source pour le spend marketing**

Deux sources décrivent la dépense : l'export campagne (`campaign_spend_export`) et le plan média (`media_plan`). Elles ne se réconcilient pas. **Règle appliquée dans dbt** :

- **Dépense observée = export campagne**, dédupliqué et converti en FCFA (`mart_budget_allocation.campaign_spend_fcfa`). C'est la base des parts de dépense et de la répartition des 15 M FCFA.
- **Plan média = référence de comparaison.** Le prévu et le facturé (`planned_budget_fcfa`, `invoiced_fcfa`) restent des colonnes distinctes, jamais additionnées à l'export. L'écart reste visible (`spend_vs_plan_ratio`, `campaign_vs_invoiced_variance_fcfa`).
- Aucune des deux sources n'est déclarée « vraie » : la règle dit seulement quelle source alimente quel calcul.

**Conséquence, à connaître :** la radio, les influenceurs et l'activation terrain sont saisis à la main dans l'export, et l'activation terrain n'y apparaît pas (0 FCFA alors que le plan indique 2,41 M FCFA facturés). Ces canaux pèsent donc moins dans la dépense observée que dans le facturé :

| Canal | Part de l'export campagne | Part du facturé (plan média) |
|---|---|---|
| Radio | 12,6 % | 26,2 % |
| Influenceurs | 5,7 % | 8,3 % |
| Activation terrain | 0,0 % | 13,8 % |

La répartition des 15 M FCFA se fait au prorata de la dépense observée (avec un socle de 1 M FCFA par canal) : elle hérite de ce biais.

**À trancher avec Kômian :** la base de répartition doit-elle rester la dépense observée (export) ou devenir le facturé (plan média) ? Ce choix change l'allocation proposée.

### **7. Ce que les données ne permettent pas de prouver**

Le jeu de données ne contient pas de parcours permettant de relier directement une exposition ou un clic publicitaire à une commande.

Il n'est donc pas possible de calculer une véritable attribution marketing par canal ni d'affirmer qu'un canal a causé une vente donnée.

Les comparaisons entre dépenses marketing et évolution des ventes seront donc présentées comme des observations temporelles ou des signaux, et non comme une preuve de causalité.

### **8. Principe de décision**

La recommandation des 15 millions FCFA combinera :

- les dépenses observées et leur répartition ;
- l'évolution des ventes ;
- les signaux issus de la voix client ;
- la qualité et la complétude des données ;
- le niveau de confiance associé à chaque conclusion.

Chaque recommandation indiquera ce qui est démontré, ce qui est seulement indicatif et ce qui devra être vérifié dans les 90 jours.

### **9. Critère de réussite**

Le processus doit permettre à Kômian de reproduire le reporting mensuel à partir des cinq sources, avec des chiffres traçables, des contrôles de qualité, une composante IA évaluée et une restitution directement exploitable par le marketing, sans intervention d'un Data Engineer.